In [1]:
no_of_threads = "12"

import os
os.environ["VECLIB_MAXIMUM_THREADS"] = no_of_threads
os.environ["OMP_NUM_THREADS"] = no_of_threads


In [2]:


import torch
from torch import linalg as LA


import time
import statistics
from typing import Tuple
from typing_extensions import Annotated
import transformers

from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import platform
import subprocess
import time
from datasets import load_dataset

In [3]:
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")

In [4]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"   

#model_hf = AutoModelForCausalLM.from_pretrained(
#    MODEL_ID, revision=REVISION, dtype=torch.float32, device_map="cpu"
#)

model_hf = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=torch.float32
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)

W1025 22:50:12.008000 65441 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [5]:
import re, time, statistics, torch

def extract_letter(text):
   
    m = re.search(r"\b([A-E1-5])\b", text)
    return m.group(1) if m else None

def time_counter(no_of_threads: int, model, tokenizer, prompts: list, results: list,
                 answer_keys: list = None, max_new_tokens: int = 128, repeats: int = 3):

    torch.set_num_threads(int(no_of_threads))
    print("threads:", torch.get_num_threads())
    time.sleep(2)

    for x in range(len(prompts)):
        messages = [{"role": "user", "content": prompts[x]}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer(text, return_tensors="pt").to(model.device)
        input_len = model_inputs["input_ids"].shape[1]

        with torch.inference_mode():
            # warm-up (not timed)
            model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)

            n_lst = []
            for i in range(repeats):
                start_time = time.perf_counter()
                out_put_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens, do_sample=False)
                end_time = time.perf_counter()
                n_lst.append(end_time - start_time)

        new_tokens = out_put_ids.shape[1] - input_len
        answer = tokenizer.decode(out_put_ids[0][input_len:], skip_special_tokens=True)
        letter = extract_letter(answer)

        mean = statistics.mean(n_lst)
        stndev = statistics.stdev(n_lst) if len(n_lst) > 1 else 0.0

        n_dict = {
            'thread': torch.get_num_threads(),
            'prompt_id': x,
            'mean': mean,
            'stdev': stndev,
            'input_tokens': input_len,
            'new_tokens': new_tokens,
            'tok_per_s': new_tokens / mean,
            'answer': answer,
            'pred': letter,
        }
        if answer_keys is not None:
            n_dict['key'] = answer_keys[x]
            n_dict['correct'] = (letter == answer_keys[x])
        results.append(n_dict)

In [6]:
def make_prompt(item):
    lines = [item["question"]]
    for label, text in zip(item["choices"]["label"], item["choices"]["text"]):
        lines.append(f"{label}. {text}")
    lines.append("Answer with the letter and briefly explain why.")
    return "\n".join(lines)





In [7]:
results = []
prompts = [make_prompt(arc[i]) for i in range(32)]
answer_keys = [arc[i]["answerKey"] for i in range(32)]

with open("prompts_arc32.json", "w") as f:
    json.dump({"prompts": prompts, "answer_keys": answer_keys}, f, indent=2)

with open("prompts_arc32.json") as f:
    data = json.load(f)
prompts, answer_keys = data["prompts"], data["answer_keys"]
print(len(prompts), len(answer_keys))   # 32 32

32 32


In [8]:
for i in range(1,13):
    time_counter(i,model_hf,tokenizer,prompts,results)

threads: 1
threads: 2
threads: 3
threads: 4
threads: 5
threads: 6
threads: 7
threads: 8
threads: 9
threads: 10
threads: 11
threads: 12


In [9]:
results

[{'thread': 1,
  'prompt_id': 0,
  'mean': 16.081495069665834,
  'stdev': 0.11286583033596818,
  'input_tokens': 100,
  'new_tokens': 128,
  'tok_per_s': 7.959458958604138,
  'answer': "The correct answer is C: Planetary days will become shorter.\n\nExplanation:\nWhen a meteorite impacts a planet, it can cause significant changes to its surface structure and composition. The increased friction from the impact could lead to more frequent collisions between rocks on the planet's surface, which would result in shorter days as the planet slows down due to these collisions. This process is known as tidal locking or synchronous rotation, where one side of the planet always faces the sun, leading to shorter days for that side compared to the other side facing away from the sun.\n\nOptions A (Planetary density will decrease) and B (Planetary years will become",
  'pred': 'C'},
 {'thread': 1,
  'prompt_id': 1,
  'mean': 14.179358360997867,
  'stdev': 0.2412655431809171,
  'input_tokens': 115,
 

In [10]:
print(torch.__config__.show())

PyTorch built with:
  - GCC 4.2
  - C++ Version: 202002
  - clang 21.1.8
  - OpenMP 202011
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: DEFAULT
  - Build settings: BLAS_INFO=open, BUILD_TYPE=Release, COMMIT_SHA=Unknown, CUDA_FLAGS= -DLIBCUDACXX_ENABLE_SIMPLIFIED_COMPLEX_OPERATIONS -Xfatbin -compress-all  -Wno-deprecated-gpu-targets --expt-extended-lambda -DCUB_WRAPPED_NAMESPACE=at_cuda_detail -DDISABLE_CUSPARSE_DEPRECATED -DCUDA_HAS_FP16=1 -D__CUDA_NO_HALF_OPERATORS__ -D__CUDA_NO_HALF_CONVERSIONS__ -D__CUDA_NO_HALF2_OPERATORS__ -D__CUDA_NO_BFLOAT16_CONVERSIONS__ -DC10_NODEPRECATED, CXX_COMPILER=/Users/runner/miniforge3/conda-bld/libtorch_1791585336308/_build_env/bin/arm64-apple-darwin20.0.0-clang++, CXX_FLAGS=-ftree-vectorize -fPIC -fstack-protector-strong -O2 -pipe -stdlib=libc++  -fmessage-length=0 -isystem /Users/runner/miniforge3/conda-bld/libtorch_1791585336308/_h_env_placehold_placehold_placehold_placehold_placehold_placehold_pla

In [ ]:
t1 = next(r['mean'] for r in results if r['thread'] == 1)

n_res = []
for r in results:
    n = r['thread']
    speedup = t1 / r['mean']
    efficiency = (speedup / n) * 100
    f = ((n / speedup - 1) / (n - 1)) * 100 if n > 1 else None   

    n_res.append({
        'threads': n,
        'mean': r['mean'],
        'speedup': speedup,
        'efficiency': efficiency,
        'serial_fraction': f,
    })

In [12]:
n_res

[{'threads': 1,
  'mean': 16.081495069665834,
  'speedup': 1.0,
  'efficiency': 100.0,
  'serial_fraction': None},
 {'threads': 1,
  'mean': 14.179358360997867,
  'speedup': 1.1341482922034072,
  'efficiency': 113.41482922034072,
  'serial_fraction': None},
 {'threads': 1,
  'mean': 9.798143250324452,
  'speedup': 1.6412798485195976,
  'efficiency': 164.12798485195975,
  'serial_fraction': None},
 {'threads': 1,
  'mean': 14.561790402658517,
  'speedup': 1.1043624873717361,
  'efficiency': 110.43624873717361,
  'serial_fraction': None},
 {'threads': 1,
  'mean': 17.20892713899957,
  'speedup': 0.9344856271267078,
  'efficiency': 93.44856271267078,
  'serial_fraction': None},
 {'threads': 1,
  'mean': 15.393534763997499,
  'speedup': 1.0446915095340767,
  'efficiency': 104.46915095340768,
  'serial_fraction': None},
 {'threads': 1,
  'mean': 14.308248180663213,
  'speedup': 1.1239317956057726,
  'efficiency': 112.39317956057727,
  'serial_fraction': None},
 {'threads': 1,
  'mean': 16.3

In [ ]:
import pandas as pd
from datetime import datetime

run_name = datetime.now().strftime("openblas_32p_%Y%m%d_%H%M")  
csv_file = f"{run_name}.csv"
print("saving to", csv_file)

results = []
for t in range(1, 13):
    time_counter(t, model_hf, tokenizer, prompts, results, answer_keys=answer_keys)
    pd.DataFrame(results).to_csv(csv_file, index=False)
    print("saved after", t, "threads,", len(results), "rows")

In [ ]:
%pip install matplotlib



In [ ]:
import matplotlib.pyplot as plt

RUN_NAME = "run4"   # change per run, used in the file names

threads = [r["threads"] for r in n_res]
speedup = [r["speedup"] for r in n_res]
efficiency = [r["efficiency"] for r in n_res]
# Amdahl serial fraction (%), skipping 1 thread
f_threads = [n for n in threads if n > 1]
serial = [((n / s) - 1) / (n - 1) * 100 for n, s in zip(threads, speedup) if n > 1]

BLUE, GREY = "#2a78d6", "#8a8980"

def tidy(ax, ylabel, title):
    ax.set_xlabel("Threads (N)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(threads)
    ax.grid(True, color="#e4e3dc")
    ax.spines[["top", "right"]].set_visible(False)

# 1. Speedup vs ideal
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(threads, threads, "--", color=GREY, label="Ideal (S = N)")
ax.plot(threads, speedup, "o-", color=BLUE, linewidth=2, label="Measured")
tidy(ax, "Speedup (T1 / TN)", "Speedup vs threads")
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(f"{RUN_NAME}_speedup.png", dpi=200); plt.show()

# 2. Efficiency
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(threads, efficiency, "o-", color=BLUE, linewidth=2)
ax.set_ylim(0, 110)
tidy(ax, "Parallel efficiency (%)", "Parallel efficiency vs threads")
fig.tight_layout(); fig.savefig(f"{RUN_NAME}_efficiency.png", dpi=200); plt.show()

# 3. Serial fraction
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(f_threads, serial, "o-", color=BLUE, linewidth=2)
ax.set_ylim(bottom=0)
tidy(ax, "Implied serial fraction f (%)", "Amdahl serial fraction vs threads")
fig.tight_layout(); fig.savefig(f"{RUN_NAME}_serial_fraction.png", dpi=200); plt.show()